In [ ]:
import numpy as np
import pandas as pd 
from sklearn.discriminant_analysis import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import(confusion_matrix, accuracy_score, precision_score, recall_score, f1_score)

# Load the data 
data_url = '../spambase_data/spambase.data'
names_url = '../spambase_data/spambase.names'

# parse the feature names from the .names file
feature_names = []
with open(names_url, "r") as f:
    for line in f:
        line = line.strip()
        if ':' in line and not line.startswith("|"):
            feature_names.append(line.split(":")[0].strip())
feature_names.append("label") # last column is the class label

df = pd.read_csv(data_url, header=None, names=feature_names)
x = df.drop("label", axis=1).values
y = df["label"].values
names = feature_names[:-1]

# Create the train test split for the data
x_tr, x_te, y_tr, y_te = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y 
)

# Train the logistic regression model
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)

model = LogisticRegression(max_iter=2000, solver="lbfgs", C=1.0, random_state=42)
model.fit(x_tr, y_tr)

# Helper function to evaluate the performance of my model
def evaluate(y_true, y_pred, label=""):
    cm = confusion_matrix(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    err = 1 - acc
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    if label:
        print(f"\n{'='*55}")
        print(f" {label}")
        print(f"{'='*55}")
    print(f"\nConfusion Matrix (rows=actual, cols=predicted):")
    print(f"            Pred 0   Pred 1")
    print(f"    Actual 0 {cm[0,0]:6d}   {cm[0,1]:6d}")
    print(f"    Actual 1 {cm[1,0]:6d}   {cm[1,1]:6d}")
    print(f"\n  Accuracy    : {acc:.4f}")
    print(f"    Error   : {err:.4f}")
    print(f"    Precision   : {prec:.4f}")
    print(f"    Recall  : {rec:.4f}")
    print(f"    F1 Score    : {f1:.4f}")
    return acc, prec, rec

y_pred = model.predict(x_te)
evaluate(y_te, y_pred, label="Part 1 test-set Performance (T = 0.50)")

print(f"\n{'='*55}")
print(" Part 2 - Feature Coefficients")
print(f"{'='*55}")

coefs = model.coef_[0]
coef_df = (pd.DataFrame({"feature": names, "coefficient": coefs})
           .sort_values("coefficient", ascending=False))

print("\nTop 10 Positive contributors (-> SPAM):")
print(coef_df.head(10).to_string(index=False))

print("\nTop 10 Negative contributors (-> Not SPAM):")
print(coef_df.tail(10).sort_values("coefficient").to_string(index=False))

print(f"\n{'='*55}")
print("     Part 3 - Varying Decision Threshold")
print(f"{'='*55}")

probs = model.predict_proba(x_te)[:, 1] # P(spam)
thresholds = [0.25, 0.50, 0.75, 0.90]

print(f"\n{'Threshold':>10} {'Accuracy':>10} {'Precision':>10} {'Recall':>10}")
print("-"*45)
for T in thresholds:
    y_pred_T = (probs >=T).astype(int)
    acc = accuracy_score(y_te, y_pred_T)
    prec = precision_score(y_te, y_pred_T, zero_division=0)
    rec = recall_score(y_te, y_pred_T, zero_division=0)
    print(f"{T:>10.2f} {acc:>10.4f} {prec:>10.4f} {rec:>10.4f}")




 Part 1 test-set Performance (T = 0.50)

Confusion Matrix (rows=actual, cols=predicted):
            Pred 0   Pred 1
    Actual 0    667       30
    Actual 1     53      401

  Accuracy    : 0.9279
    Error   : 0.0721
    Precision   : 0.9304
    Recall  : 0.8833
    F1 Score    : 0.9062

 Part 2 - Feature Coefficients

Top 10 Positive contributors (-> SPAM):
                   feature  coefficient
capital_run_length_longest     1.719832
               char_freq_$     1.366262
capital_run_length_average     1.185180
               char_freq_#     0.988575
              word_freq_3d     0.867262
          word_freq_remove     0.831901
             word_freq_000     0.793984
            word_freq_free     0.753701
        word_freq_business     0.406168
             word_freq_our     0.389064

Top 10 Negative contributors (-> Not SPAM):
          feature  coefficient
 word_freq_george    -3.899146
     word_freq_hp    -2.294340
     word_freq_85    -1.692253
     word_freq_cs    -1.62

# comments on the different decision thresholds
- The decsion threshold controls the trade-off between precision and recall. As T increases, the model becomes more conservative, it only labels emails as spam when it is highly confident, which leads to fewer false positives and higher precision. However, this means more actual spam emails slip through undetected, causing recall to drop. Conversly, lowering the T makes the model more aggresive in flagging spam, catching more true positives (high recall) at the cost of also flagging more legitamate email (lower precision).
- The best performance of the model (in terms of balanced recall, precision and accuracy) occurs at the T = .5 threshold which is the natural threshold for logistic regression allowing the model to simply choose which class is more probable.